# Fractional delayed impulsive fault model — study + real fault map (Colab)

Run the cells top to bottom. **Everything (results, figures, tables, Excel) is saved directly in your Google Drive folder `MyDrive/fault_study/`**, so nothing is lost if Colab disconnects, and the study resumes where it stopped (finished replicates are skipped).

- `MODE='QUICK'` = fast sanity check; `MODE='FULL'` = study of the article (~1.5 h on 2 CPUs).
- The real-fault map uses `cartopy` and the GEM Global Active Faults Database (downloaded from GitHub).

In [ ]:
# 1. Google Drive: all outputs are written here
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/fault_study'
for d in ('out','figs','tabs','data'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
os.makedirs('/content/stats', exist_ok=True)
%cd /content/stats
for d in ('out','figs','tabs','data'):
    !ln -sfn {DRIVE}/{d} {d}
print('outputs ->', DRIVE)

In [ ]:
# 2. dependencies
!pip -q install numba openpyxl
!apt-get -qq install -y libgeos-dev poppler-utils >/dev/null
!pip -q install cartopy
import multiprocessing
print('CPUs:', multiprocessing.cpu_count())

In [ ]:
%%writefile recovery_study.py
"""
Parameter-recovery study for the corrected fractional-delay-impulsive fault model.

MODEL (two faults, i = 0, 1; time in years, stress in arbitrary units)
  Caputo order alpha (shared):  D^a x_i = v_i - beta x_i + gamma (x_j - x_i) + sigma_x * white noise
  Soft threshold (hazard):      lambda_i = lam0 * exp( b (x_i - theta) ),  lam0 = 1 (fixed, else redundant with theta)
  Rupture of fault i:           x_i -> x_i - Delta,  Delta = theta (1 - rho)
  Delayed pulse on the other:   + gamma * C * Delta   arriving tau = d / c years later   (delay on pulses ONLY)
  Loading from reference time:  v_i = beta theta (1 - rho E_a(-beta Tbar_i^a)) / (1 - E_a(-beta Tbar_i^a))
                                (Tbar_i = first-passage time of the ISOLATED fault started at rho*theta; it is a
                                 reference time, not the realised mean recurrence, because memory persists)
  Observations:                 rupture times (on the time grid) and a noisy GNSS-like proxy y = x + eps every year.

ESTIMATED (4 global parameters): alpha, gamma, c, b.   FIXED: beta, theta, rho, Tbar_i, d, C, sigma_x, sigma_y.

DATA are generated with the EXACT memory kernel (product integration, O(n^2)); INFERENCE uses the Markov
sum-of-exponentials (SOE) representation of the kernel t^(a-1)/Gamma(a) (diffusive representation
   t^(a-1)/G(a) = sin(pi a)/pi * int_0^inf s^(-a) e^(-s t) ds, discretised on a log grid, plus a constant tail term)
inside a bootstrap particle filter, so the study also tests the approximation (no "inverse crime" in the memory).
Both use the same time grid h, same hazard model, so the soft-threshold/grid discretisation is shared.

Usage
  python recovery_study.py                run checks, one dataset, likelihood slices, MLE        (several minutes)
  python recovery_study.py --quick        smaller / faster version
  python recovery_study.py --checks       numerical sanity checks only (seconds)
  python recovery_study.py --profile      add the profile deviance over alpha (tests H0: alpha = 1)  (slow)
  python recovery_study.py --reps 5       repeat data generation + MLE over 5 datasets                 (slow)
Requires numpy, scipy, matplotlib.
"""
import os
import sys
import time
from math import gamma as Gamma, lgamma, pi, sin

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize

FIXED = dict(beta=0.05, theta=10.0, rho=0.2, Tbar=np.array([40.0, 45.0]), d=30.0, C=3.0,
             sigx=0.10, sigy=0.30, lam0=1.0)
TRUE = dict(alpha=0.7, gamma=0.08, c=10.0, b=3.0)          # c in km/yr, tau = d/c = 3 yr


# ----------------------------------------------------------------------------------------------
# Building blocks
# ----------------------------------------------------------------------------------------------
def mittag_leffler(alpha, z, terms=250):
    return sum(z ** k / np.exp(lgamma(alpha * k + 1.0)) for k in range(terms))


def loading_rates(alpha, fx=FIXED):
    """v_i from the reference first-passage times (closed form)."""
    E = np.array([mittag_leffler(alpha, -fx["beta"] * T ** alpha) for T in fx["Tbar"]])
    return fx["beta"] * fx["theta"] * (1.0 - fx["rho"] * E) / (1.0 - E)


def soe(alpha, h, T, du=1.0):
    """Sum-of-exponentials weights/rates for k(t) = t^(a-1)/Gamma(a), valid for t in [h, T].
    Returns (s, w): s[0] = 0 carries the slow tail (pure integrator)."""
    if alpha >= 1.0 - 1e-12:
        return np.array([0.0]), np.array([1.0])
    umin, umax = np.log(0.01 / T), np.log(1000.0 / h)
    u = np.arange(umin, umax + du, du)
    s = np.exp(u)
    w = sin(pi * alpha) / pi * np.exp((1.0 - alpha) * u) * du
    w0 = sin(pi * alpha) / (pi * (1.0 - alpha)) * np.exp((1.0 - alpha) * (umin - du / 2.0))
    return np.concatenate([[0.0], s]), np.concatenate([[w0], w])


def cell_weights_exact(alpha, h, n):
    m = np.arange(1, n + 1, dtype=float)
    return h ** alpha * (m ** alpha - (m - 1.0) ** alpha) / Gamma(alpha + 1.0)


def cell_weights_soe(alpha, h, n, T):
    s, w = soe(alpha, h, T)
    lag = np.arange(0, n)[:, None] * h
    gain = np.where(s > 0, w * (1 - np.exp(-s * h)) / np.where(s > 0, s, 1.0), w * h)
    return (gain[None, :] * np.exp(-s[None, :] * lag)).sum(1)


# ----------------------------------------------------------------------------------------------
# Data generator (exact memory)
# ----------------------------------------------------------------------------------------------
def generate(par, fx=FIXED, T=800.0, h=0.25, seed=0, obs_every_years=1.0):
    rng = np.random.default_rng(seed)
    a, g, c, bh = par["alpha"], par["gamma"], par["c"], par["b"]
    n = int(round(T / h))
    v = loading_rates(a, fx)
    ds = max(int(round(fx["d"] / c / h)), 1)
    m = np.arange(n + 2, dtype=float)
    bw = m ** a - (m - 1.0) ** a
    bw[0] = 0.0
    coef = h ** a / Gamma(a + 1.0)
    x0 = np.full(2, fx["rho"] * fx["theta"])
    drop = fx["theta"] * (1.0 - fx["rho"])
    x = np.zeros((n + 1, 2)); x[0] = x0
    Frev = np.zeros((2, n + 1))
    J = np.zeros(2)
    pend = {}
    ev = np.zeros((n + 1, 2), dtype=bool)
    for k in range(n):
        F = v - fx["beta"] * x[k] + g * (x[k, ::-1] - x[k]) + fx["sigx"] * rng.standard_normal(2) / np.sqrt(h)
        Frev[:, n - k] = F
        mem = Frev[:, n - k:n + 1] @ bw[1:k + 2]
        if (k + 1) in pend:
            J += pend.pop(k + 1)
        xn = x0 + coef * mem + J
        lam = fx["lam0"] * np.exp(np.clip(bh * (xn - fx["theta"]), -30, 10))
        hit = rng.random(2) < 1.0 - np.exp(-lam * h)
        for i in range(2):
            if hit[i]:
                J[i] -= drop
                xn[i] -= drop
                ev[k + 1, i] = True
                if k + 1 + ds <= n:
                    q = pend.setdefault(k + 1 + ds, np.zeros(2))
                    q[1 - i] += g * fx["C"] * drop
        x[k + 1] = xn
    oe = max(int(round(obs_every_years / h)), 1)
    obs_steps = np.arange(oe, n + 1, oe)
    y = x[obs_steps] + fx["sigy"] * rng.standard_normal((len(obs_steps), 2))
    return dict(T=T, h=h, n=n, x=x, ev=ev, obs_steps=obs_steps, y=y, par=dict(par))


# ----------------------------------------------------------------------------------------------
# Bootstrap particle filter with the SOE (Markov) representation
# ----------------------------------------------------------------------------------------------
_NOISE = {}


def _noise(n, P, seed):
    key = (n, P, seed)
    if key not in _NOISE:
        r = np.random.default_rng(10_000 + seed)
        _NOISE[key] = (r.standard_normal((n, P, 2)), r.random(n))
    return _NOISE[key]


def pf_loglik(par, data, fx=FIXED, npart=150, seed=0):
    a, g, c, bh = par["alpha"], par["gamma"], par["c"], par["b"]
    h, n, T = data["h"], data["n"], data["T"]
    P = npart
    v = loading_rates(a, fx)
    s, w = soe(a, h, T)
    decay = np.exp(-s * h)
    gain = np.where(s > 0, w * (1 - decay) / np.where(s > 0, s, 1.0), w * h)
    ds = max(int(round(fx["d"] / c / h)), 1)
    D = ds + 2
    pend = np.zeros((D, P, 2))
    z = np.zeros((P, 2, len(s)))
    J = np.zeros((P, 2))
    x0 = fx["rho"] * fx["theta"]
    drop = fx["theta"] * (1.0 - fx["rho"])
    xi, uu = _noise(n, P, seed)
    obs_at = {int(k): i for i, k in enumerate(data["obs_steps"])}
    ev, y = data["ev"], data["y"]
    W = np.full(P, 1.0 / P)
    ll = 0.0
    c_obs = -np.log(fx["sigy"] * np.sqrt(2 * np.pi))
    for k in range(n):
        xcur = x0 + z.sum(-1) + J
        F = v - fx["beta"] * xcur + g * (xcur[:, ::-1] - xcur) + fx["sigx"] * xi[k] / np.sqrt(h)
        z = decay * z + gain * F[..., None]
        slot = (k + 1) % D
        J += pend[slot]
        pend[slot] = 0.0
        x = x0 + z.sum(-1) + J
        lam = fx["lam0"] * np.exp(np.clip(bh * (x - fx["theta"]), -30, 10))
        p = np.clip(1.0 - np.exp(-lam * h), 1e-12, 1 - 1e-12)
        e = ev[k + 1]
        logw = np.where(e, np.log(p), np.log1p(-p)).sum(1)
        for i in range(2):
            if e[i]:
                J[:, i] -= drop
                x[:, i] -= drop
                pend[(k + 1 + ds) % D, :, 1 - i] += g * fx["C"] * drop
        if (k + 1) in obs_at:
            r = (y[obs_at[k + 1]] - x) / fx["sigy"]
            logw += (-0.5 * r * r + c_obs).sum(1)
        mx = logw.max()
        ww = W * np.exp(logw - mx)
        sw = ww.sum()
        ll += mx + np.log(sw)
        W = ww / sw
        if 1.0 / np.sum(W * W) < 0.5 * P:
            idx = np.minimum(np.searchsorted(np.cumsum(W), (np.arange(P) + uu[k]) / P), P - 1)
            z, J, pend = z[idx], J[idx], pend[:, idx, :]
            W = np.full(P, 1.0 / P)
    return ll


# ----------------------------------------------------------------------------------------------
# Parameter transforms and optimisation
# ----------------------------------------------------------------------------------------------
def to_par(theta):
    a = 1.0 / (1.0 + np.exp(-theta[0]))
    return dict(alpha=min(a, 0.999), gamma=np.exp(theta[1]), c=np.exp(theta[2]), b=np.exp(theta[3]))


def from_par(par):
    a = par["alpha"]
    return np.array([np.log(a / (1 - a)), np.log(par["gamma"]), np.log(par["c"]), np.log(par["b"])])


def fit_mle(data, start, npart=150, seed=0, maxiter=120, fixed_alpha=None, fx=FIXED):
    """Nelder-Mead on (logit alpha, log gamma, log c, log b); optionally alpha held fixed."""
    n_eval = [0]

    def negll(th):
        par = to_par(th) if fixed_alpha is None else dict(alpha=fixed_alpha, gamma=np.exp(th[0]), c=np.exp(th[1]), b=np.exp(th[2]))
        n_eval[0] += 1
        try:
            return -pf_loglik(par, data, fx, npart, seed)
        except FloatingPointError:
            return 1e9

    th = from_par(start) if fixed_alpha is None else np.log([start["gamma"], start["c"], start["b"]])
    best = np.inf
    for _ in range(4):                      # restart Nelder-Mead until it stops improving (rough PF surface)
        res = minimize(negll, th, method="Nelder-Mead",
                       options=dict(maxiter=maxiter, xatol=1e-2, fatol=0.02, adaptive=True))
        th = res.x
        if best - res.fun < 0.5:
            best = min(best, res.fun)
            break
        best = res.fun
    par = to_par(th) if fixed_alpha is None else dict(alpha=fixed_alpha, gamma=np.exp(th[0]), c=np.exp(th[1]), b=np.exp(th[2]))
    return par, -best, n_eval[0]


# ----------------------------------------------------------------------------------------------
# Checks
# ----------------------------------------------------------------------------------------------
def run_checks(h=0.25, T=800.0):
    print("CHECK 1: loading rate from reference first-passage time (isolated deterministic fault)")
    fx = dict(FIXED, sigx=0.0, sigy=1.0)
    for a in (1.0, 0.7, 0.5):
        v = loading_rates(a, fx)
        beta, theta, rho = fx["beta"], fx["theta"], fx["rho"]
        n = int(round(120 / 0.05))
        m = np.arange(n + 2, dtype=float); bw = m ** a - (m - 1.0) ** a; bw[0] = 0
        coef = 0.05 ** a / Gamma(a + 1.0); Frev = np.zeros(n + 1); x = np.zeros(n + 1); x[0] = rho * theta
        t_hit = np.nan
        for k in range(n):
            Frev[n - k] = v[0] - beta * x[k]
            x[k + 1] = x[0] + coef * (Frev[n - k:n + 1] @ bw[1:k + 2])
            if x[k + 1] >= theta:
                t_hit = (k + 1) * 0.05
                break
        print(f"   alpha={a}: v={v[0]:.3f}, first passage {t_hit:.2f} yr (target Tbar={fx['Tbar'][0]:.0f})")

    print("CHECK 2: SOE kernel approximation, max relative error of cell weights over lags in [h, T]")
    n = int(round(T / h))
    for a in (0.5, 0.7, 0.9, 0.97, 1.0):
        ex = cell_weights_exact(a, h, n)
        ap = cell_weights_soe(a, h, n, T)
        rel = np.abs(ap - ex) / ex
        print(f"   alpha={a}: terms={len(soe(a, h, T)[0])}, error newest cell={rel[0]:.3%}, "
              f"error lags>=2 cells: max={rel[1:].max():.3%}")

    print("CHECK 3: SOE particle-filter dynamics vs exact-memory path (no noise, no events)")
    fx0 = dict(FIXED, sigx=0.0, lam0=0.0)                               # lam0 = 0 => no ruptures
    par = dict(TRUE)
    data = generate(par, fx0, T=200.0, h=h, seed=1)
    a = par["alpha"]; v = loading_rates(a, fx0); s, w = soe(a, h, 200.0)
    decay = np.exp(-s * h); gain = np.where(s > 0, w * (1 - decay) / np.where(s > 0, s, 1.0), w * h)
    z = np.zeros((2, len(s))); xs = np.zeros((data["n"] + 1, 2)); x0 = fx0["rho"] * fx0["theta"]; xs[0] = x0
    for k in range(data["n"]):
        xc = x0 + z.sum(-1)
        F = v - fx0["beta"] * xc + par["gamma"] * (xc[::-1] - xc)
        z = decay * z + gain * F[:, None]
        xs[k + 1] = x0 + z.sum(-1)
    print(f"   events in generated data: {int(data['ev'].sum())};  max |x_SOE - x_exact| = "
          f"{np.abs(xs - data['x']).max():.4f} (stress units, theta={fx0['theta']})")


# ----------------------------------------------------------------------------------------------
# Experiments
# ----------------------------------------------------------------------------------------------
def slices(data, npart, seed, outdir):
    grids = dict(alpha=[0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9, 0.97],
                 gamma=[0.0, 0.02, 0.05, 0.08, 0.12, 0.2],
                 c=[2.5, 5.0, 7.5, 10.0, 15.0, 30.0, 60.0],
                 b=[1.0, 2.0, 3.0, 4.0, 6.0, 10.0])
    out = {}
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.6), sharey=False)
    for a_, (name, grid) in zip(ax, grids.items()):
        vals = []
        for gv in grid:
            par = dict(TRUE); par[name] = max(gv, 1e-4) if name == "gamma" else gv
            vals.append(pf_loglik(par, data, FIXED, npart, seed))
        vals = np.array(vals); out[name] = (grid, vals)
        a_.plot(grid, np.maximum(vals - vals.max(), -150), "o-")
        a_.set_ylim(-160, 5)                      # values below -150 are clipped (model strongly rejected)
        a_.axvline(TRUE[name], color="r", ls=":", label="truth")
        a_.set_title(f"slice in {name}"); a_.set_xlabel(name); a_.set_ylabel("log-lik minus max")
        if name in ("c",):
            a_.set_xscale("log")
        a_.legend()
    fig.suptitle("Particle-filter log-likelihood slices (other parameters at truth)", y=1.03)
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "fig_recovery_slices.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    return out


def plot_data(data, outdir):
    fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
    t = np.arange(data["n"] + 1) * data["h"]
    for i in range(2):
        ax[i].plot(t, data["x"][:, i], lw=0.8, label="latent stress")
        ax[i].plot(data["obs_steps"] * data["h"], data["y"][:, i], ".", ms=2, alpha=0.5, label="GNSS-like obs")
        te = t[data["ev"][:, i]]
        ax[i].plot(te, np.full(len(te), FIXED["theta"]), "rv", ms=4, label="rupture")
        ax[i].set_ylabel(f"fault {i + 1}")
    ax[0].legend(loc="upper right", ncol=3, fontsize=8); ax[1].set_xlabel("years")
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "fig_data.png"), dpi=140); plt.close(fig)


def main():
    quick = "--quick" in sys.argv
    reps = int(sys.argv[sys.argv.index("--reps") + 1]) if "--reps" in sys.argv else 1
    outdir = "out"; os.makedirs(outdir, exist_ok=True)
    t0 = time.time()
    run_checks()
    if "--checks" in sys.argv:
        return
    T = 400.0 if quick else 800.0
    npart = 100 if quick else 150
    h = 0.25
    print(f"\nTRUE parameters: {TRUE}  (tau = d/c = {FIXED['d'] / TRUE['c']:.1f} yr);  T={T:.0f} yr, h={h}, particles={npart}")
    data = generate(TRUE, FIXED, T=T, h=h, seed=11)
    ne = data["ev"].sum(0)
    print(f"generated ruptures: fault1={ne[0]}, fault2={ne[1]}  (reference Tbar = {FIXED['Tbar']})")
    plot_data(data, outdir)

    t1 = time.time(); ll_true = pf_loglik(TRUE, data, FIXED, npart, 0)
    print(f"PF log-likelihood at truth: {ll_true:.1f}   ({time.time() - t1:.1f} s per evaluation)")
    spread = [pf_loglik(TRUE, data, FIXED, npart, sd) for sd in range(1, 6)]
    print(f"Monte-Carlo sd of the PF log-likelihood over seeds: {np.std(spread):.2f}")

    sl = slices(data, npart, 0, outdir)
    for name, (grid, vals) in sl.items():
        print(f"slice {name}: argmax at {grid[int(np.argmax(vals))]}   (truth {TRUE[name]});  "
              f"drop at ends: {vals.max() - vals[0]:.1f} / {vals.max() - vals[-1]:.1f}")

    start = dict(alpha=0.85, gamma=0.04, c=5.0, b=2.0)
    print("\nMLE from a perturbed start (Nelder-Mead on the PF likelihood, common random numbers):")
    rows = []
    for r in range(reps):
        dr = data if r == 0 else generate(TRUE, FIXED, T=T, h=h, seed=11 + r)
        par, ll, ne_ = fit_mle(dr, start, npart, 0, maxiter=60 if quick else 120)
        rows.append([par["alpha"], par["gamma"], par["c"], par["b"], ll])
        print(f"  rep {r}: alpha={par['alpha']:.2f}  gamma={par['gamma']:.3f}  c={par['c']:.1f}  b={par['b']:.2f}  "
              f"loglik={ll:.1f}  ({ne_} evals)")
    rows = np.array(rows)
    np.savetxt(os.path.join(outdir, "mle_estimates.csv"), rows, delimiter=",", header="alpha,gamma,c,b,loglik", comments="")
    if reps > 1:
        print("  mean:", np.round(rows.mean(0)[:4], 3), " sd:", np.round(rows.std(0, ddof=1)[:4], 3),
              " truth:", [TRUE[k] for k in ("alpha", "gamma", "c", "b")])

    # Robustness: the loading rates come from the reference times Tbar_i, which on real data are only approximately
    # known.  Refit with Tbar misspecified by +25% (data still generated with the true Tbar).
    fx_bad = dict(FIXED, Tbar=FIXED["Tbar"] * 1.25)
    par_b, ll_b, ne_b = fit_mle(data, start, npart, 0, maxiter=60 if quick else 120, fx=fx_bad)
    print(f"\nMLE with Tbar misspecified (+25%):  alpha={par_b['alpha']:.2f}  gamma={par_b['gamma']:.3f}  "
          f"c={par_b['c']:.1f}  b={par_b['b']:.2f}  loglik={ll_b:.1f}  ({ne_b} evals)   [truth alpha=0.7]")

    if "--profile" in sys.argv:
        print("\nProfile deviance over alpha (gamma, c, b re-optimised); H0: alpha = 1")
        grid = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        pr = []
        for av in grid:
            par, ll, _ = fit_mle(data, dict(TRUE), npart, 0, maxiter=60, fixed_alpha=min(av, 0.999) if av < 1 else 1.0)
            pr.append(ll); print(f"   alpha={av}: max loglik={ll:.1f}")
        pr = np.array(pr)
        plt.figure(figsize=(5, 3.4)); plt.plot(grid, 2 * (pr.max() - pr), "o-"); plt.axvline(TRUE["alpha"], color="r", ls=":")
        plt.xlabel("alpha"); plt.ylabel("profile deviance"); plt.tight_layout()
        plt.savefig(os.path.join(outdir, "fig_profile_alpha.png"), dpi=140); plt.close()
    print(f"\ndone in {time.time() - t0:.0f}s; figures and CSV in ./{outdir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fastpf.py
"""
Numba-accelerated bootstrap particle filter for the fractional delayed-impulsive fault model.
Same model, same discretisation and same resampling rule as recovery_study.pf_loglik (checked numerically),
plus a forecasting filter that returns window rupture probabilities at yearly origins.
"""
import sys
import numpy as np
import numba as nb

sys.path.insert(0, ".")
import recovery_study as rs  # noqa: E402

FIXED, TRUE = rs.FIXED, rs.TRUE


@nb.njit(cache=True, fastmath=False)
def _pf(decay, gain, v, beta, g, theta, rho, C, sigx, lam0, bh, h, n, ds, xi, uu, ev, obs_idx, y, sigy,
        P, fc_every, fc_steps, fc_K, rng_u, out_fc):
    M = decay.shape[0]
    D = ds + 2
    x0 = rho * theta
    drop = theta * (1.0 - rho)
    z = np.zeros((P, 2, M))
    J = np.zeros((P, 2))
    pend = np.zeros((D, P, 2))
    W = np.full(P, 1.0 / P)
    ll = 0.0
    c_obs = -np.log(sigy * np.sqrt(2.0 * np.pi))
    sh = np.sqrt(h)
    x = np.zeros((P, 2))
    logw = np.zeros(P)
    ww = np.zeros(P)
    z2 = np.zeros((P, 2, M))
    J2 = np.zeros((P, 2))
    pend2 = np.zeros((D, P, 2))
    cs = np.zeros(P)
    idx = np.zeros(P, dtype=np.int64)
    fc_row = 0
    for k in range(n):
        for p in range(P):
            xc0 = x0 + J[p, 0]
            xc1 = x0 + J[p, 1]
            for m in range(M):
                xc0 += z[p, 0, m]
                xc1 += z[p, 1, m]
            F0 = v[0] - beta * xc0 + g * (xc1 - xc0) + sigx * xi[k, p, 0] / sh
            F1 = v[1] - beta * xc1 + g * (xc0 - xc1) + sigx * xi[k, p, 1] / sh
            for m in range(M):
                z[p, 0, m] = decay[m] * z[p, 0, m] + gain[m] * F0
                z[p, 1, m] = decay[m] * z[p, 1, m] + gain[m] * F1
        slot = (k + 1) % D
        for p in range(P):
            J[p, 0] += pend[slot, p, 0]
            J[p, 1] += pend[slot, p, 1]
            pend[slot, p, 0] = 0.0
            pend[slot, p, 1] = 0.0
        e0 = ev[k + 1, 0]
        e1 = ev[k + 1, 1]
        for p in range(P):
            s0 = x0 + J[p, 0]
            s1 = x0 + J[p, 1]
            for m in range(M):
                s0 += z[p, 0, m]
                s1 += z[p, 1, m]
            x[p, 0] = s0
            x[p, 1] = s1
            lw = 0.0
            for i in range(2):
                arg = bh * (x[p, i] - theta)
                if arg < -30.0:
                    arg = -30.0
                if arg > 10.0:
                    arg = 10.0
                lam = lam0 * np.exp(arg)
                pr = 1.0 - np.exp(-lam * h)
                if pr < 1e-12:
                    pr = 1e-12
                if pr > 1.0 - 1e-12:
                    pr = 1.0 - 1e-12
                ei = e0 if i == 0 else e1
                if ei:
                    lw += np.log(pr)
                else:
                    lw += np.log1p(-pr)
            logw[p] = lw
        tgt = (k + 1 + ds) % D
        if e0:
            for p in range(P):
                J[p, 0] -= drop
                x[p, 0] -= drop
                pend[tgt, p, 1] += g * C * drop
        if e1:
            for p in range(P):
                J[p, 1] -= drop
                x[p, 1] -= drop
                pend[tgt, p, 0] += g * C * drop
        oi = obs_idx[k + 1]
        if oi >= 0:
            for p in range(P):
                for i in range(2):
                    r = (y[oi, i] - x[p, i]) / sigy
                    logw[p] += -0.5 * r * r + c_obs
        mx = -1e300
        for p in range(P):
            if logw[p] > mx:
                mx = logw[p]
        sw = 0.0
        for p in range(P):
            ww[p] = W[p] * np.exp(logw[p] - mx)
            sw += ww[p]
        ll += mx + np.log(sw)
        s2 = 0.0
        for p in range(P):
            W[p] = ww[p] / sw
            s2 += W[p] * W[p]
        if 1.0 / s2 < 0.5 * P:
            acc = 0.0
            for p in range(P):
                acc += W[p]
                cs[p] = acc
            j = 0
            for p in range(P):
                u = (p + uu[k]) / P
                while j < P - 1 and cs[j] < u:
                    j += 1
                idx[p] = j
            for p in range(P):
                q = idx[p]
                for i in range(2):
                    J2[p, i] = J[q, i]
                    for m in range(M):
                        z2[p, i, m] = z[q, i, m]
                for d in range(D):
                    pend2[d, p, 0] = pend[d, q, 0]
                    pend2[d, p, 1] = pend[d, q, 1]
            for p in range(P):
                for i in range(2):
                    J[p, i] = J2[p, i]
                    for m in range(M):
                        z[p, i, m] = z2[p, i, m]
                for d in range(D):
                    pend[d, p, 0] = pend2[d, p, 0]
                    pend[d, p, 1] = pend2[d, p, 1]
                W[p] = 1.0 / P
        # ---- forecast at yearly origins (no new data): forward Monte Carlo from the filtering particles
        if fc_every > 0 and (k + 1) % fc_every == 0 and fc_row < out_fc.shape[0]:
            hit0 = 0.0
            hit1 = 0.0
            for p in range(P):
                for kk in range(fc_K):
                    zt = z[p].copy()
                    Jt = J[p].copy()
                    pt = pend[:, p, :].copy()
                    h0 = False
                    h1 = False
                    for s in range(fc_steps):
                        kq = k + 1 + s
                        xc0 = x0 + Jt[0]
                        xc1 = x0 + Jt[1]
                        for m in range(M):
                            xc0 += zt[0, m]
                            xc1 += zt[1, m]
                        # deterministic forward drift plus process noise
                        ri = rng_u[fc_row, p, kk, s, 0]
                        rj = rng_u[fc_row, p, kk, s, 1]
                        F0 = v[0] - beta * xc0 + g * (xc1 - xc0) + sigx * ri / sh
                        F1 = v[1] - beta * xc1 + g * (xc0 - xc1) + sigx * rj / sh
                        for m in range(M):
                            zt[0, m] = decay[m] * zt[0, m] + gain[m] * F0
                            zt[1, m] = decay[m] * zt[1, m] + gain[m] * F1
                        sl = (kq + 1) % D
                        Jt[0] += pt[sl, 0]
                        Jt[1] += pt[sl, 1]
                        pt[sl, 0] = 0.0
                        pt[sl, 1] = 0.0
                        s0 = x0 + Jt[0]
                        s1 = x0 + Jt[1]
                        for m in range(M):
                            s0 += zt[0, m]
                            s1 += zt[1, m]
                        for i in range(2):
                            sx = s0 if i == 0 else s1
                            arg = bh * (sx - theta)
                            if arg < -30.0:
                                arg = -30.0
                            if arg > 10.0:
                                arg = 10.0
                            lam = lam0 * np.exp(arg)
                            pr = 1.0 - np.exp(-lam * h)
                            uu2 = rng_u[fc_row, p, kk, s, 2 + i]
                            if uu2 < pr:
                                Jt[i] -= drop
                                if i == 0:
                                    h0 = True
                                    pt[(kq + 1 + ds) % D, 1] += g * C * drop
                                else:
                                    h1 = True
                                    pt[(kq + 1 + ds) % D, 0] += g * C * drop
                    if h0:
                        hit0 += W[p] / fc_K
                    if h1:
                        hit1 += W[p] / fc_K
            out_fc[fc_row, 0] = k + 1
            out_fc[fc_row, 1] = hit0
            out_fc[fc_row, 2] = hit1
            fc_row += 1
    return ll


_NOISE = {}


def _noise(n, P, seed):
    key = (n, P, seed)
    if key not in _NOISE:
        r = np.random.default_rng(10_000 + seed)
        _NOISE[key] = (r.standard_normal((n, P, 2)), r.random(n))
    return _NOISE[key]


def prepare(data):
    n = data["n"]
    obs_idx = np.full(n + 1, -1, dtype=np.int64)
    obs_idx[data["obs_steps"]] = np.arange(len(data["obs_steps"]))
    return dict(ev=np.ascontiguousarray(data["ev"].astype(np.uint8)), obs_idx=obs_idx,
                y=np.ascontiguousarray(data["y"], dtype=np.float64))


def _coef(par, data, fx):
    a = par["alpha"]
    s, w = rs.soe(a, data["h"], data["T"])
    decay = np.exp(-s * data["h"])
    gain = np.where(s > 0, w * (1 - decay) / np.where(s > 0, s, 1.0), w * data["h"])
    v = rs.loading_rates(a, fx)
    ds = max(int(round(fx["d"] / par["c"] / data["h"])), 1)
    return decay, gain, v, ds


_DUMMY = np.zeros((1, 1, 1, 1, 4))
_DUMMY_OUT = np.zeros((1, 3))


def loglik(par, data, prep, fx=FIXED, npart=200, seed=0):
    decay, gain, v, ds = _coef(par, data, fx)
    xi, uu = _noise(data["n"], npart, seed)
    return _pf(decay, gain, v, fx["beta"], par["gamma"], fx["theta"], fx["rho"], fx["C"], fx["sigx"], fx["lam0"],
               par["b"], data["h"], data["n"], ds, xi, uu, prep["ev"], prep["obs_idx"], prep["y"], fx["sigy"],
               npart, 0, 0, 1, _DUMMY, _DUMMY_OUT)


def forecast(par, data, prep, fx=FIXED, npart=100, K=4, window_years=5.0, every_years=1.0, seed=0, first_origin=10.0):
    """Probability that each fault ruptures in (t, t+window] given data up to t, at yearly origins."""
    h = data["h"]
    decay, gain, v, ds = _coef(par, data, fx)
    xi, uu = _noise(data["n"], npart, seed)
    fc_every = int(round(every_years / h))
    fc_steps = int(round(window_years / h))
    n_orig = data["n"] // fc_every
    rng = np.random.default_rng(777 + seed)
    ru = np.empty((n_orig, npart, K, fc_steps, 4))
    ru[..., 0:2] = rng.standard_normal((n_orig, npart, K, fc_steps, 2))
    ru[..., 2:4] = rng.random((n_orig, npart, K, fc_steps, 2))
    out = np.zeros((n_orig, 3))
    _pf(decay, gain, v, fx["beta"], par["gamma"], fx["theta"], fx["rho"], fx["C"], fx["sigx"], fx["lam0"],
        par["b"], h, data["n"], ds, xi, uu, prep["ev"], prep["obs_idx"], prep["y"], fx["sigy"],
        npart, fc_every, fc_steps, K, ru, out)
    t = out[:, 0] * h
    keep = t >= first_origin
    return t[keep], out[keep, 1:3]


if __name__ == "__main__":
    import time
    data = rs.generate(TRUE, T=400.0, h=0.25, seed=11)
    prep = prepare(data)
    t0 = time.time()
    ref = rs.pf_loglik(TRUE, data, FIXED, npart=100, seed=0)
    t1 = time.time()
    fast = loglik(TRUE, data, prep, npart=100, seed=0)       # includes JIT compile
    t2 = time.time()
    fast2 = loglik(TRUE, data, prep, npart=100, seed=0)
    t3 = time.time()
    print(f"numpy {ref:.4f} ({t1-t0:.2f}s)   numba {fast:.4f} (compile+run {t2-t1:.2f}s)   run {fast2:.4f} ({t3-t2:.3f}s)")
    for P in (100, 200, 400):
        vals = [loglik(TRUE, data, prep, npart=P, seed=s) for s in range(12)]
        print(f"P={P}: mean {np.mean(vals):.2f} sd {np.std(vals, ddof=1):.2f}")
    tt = time.time()
    tf, pr = forecast(TRUE, data, prep, npart=100, K=4)
    print("forecast", pr.shape, f"{time.time()-tt:.2f}s", pr[:3], pr.mean(0))


In [ ]:
%%writefile stats_study.py
"""
Statistical study for the fractional delayed-impulsive fault model.
 task 'alt'  : replicate under the alternative (true alpha = 0.7): full MLE, restricted MLEs (alpha=1; gamma=0),
               profile likelihood over alpha, out-of-sample forecast scores against baselines.
 task 'null' : replicate under H0 (alpha = 1): full and restricted (alpha=1) MLE -> null law of the LR statistic.
Results: one JSON (+ NPZ for forecasts) per replicate in the output folder (resumable).
"""
import argparse
import json
import os
import sys
import time
import warnings
from multiprocessing import Pool

import numpy as np
from scipy.optimize import minimize

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import fastpf as fp  # noqa: E402

rs = fp.rs
warnings.filterwarnings("ignore")
FIXED, TRUE = fp.FIXED, fp.TRUE
NAMES = ["alpha", "gamma", "c", "b"]
TAG = "0.3"
STARTS = [dict(alpha=0.85, gamma=0.04, c=5.0, b=2.0), dict(alpha=0.5, gamma=0.15, c=20.0, b=5.0)]


def enc(name, val):
    return np.log(val / (1 - val)) if name == "alpha" else np.log(val)


def dec(name, th):
    return min(1 / (1 + np.exp(-th)), 0.999) if name == "alpha" else float(np.exp(th))


def fit(data, prep, free, base, starts, npart, maxiter=200):
    """Nelder-Mead on the particle-filter log-likelihood over the parameters in `free`; others fixed at `base`."""
    best = None
    ne = [0]

    def negll(th):
        par = dict(base)
        for nm, t in zip(free, th):
            par[nm] = dec(nm, t)
        ne[0] += 1
        try:
            v = fp.loglik(par, data, prep, FIXED, npart, 0)
        except Exception:
            return 1e9
        return -v if np.isfinite(v) else 1e9

    for s in starts:
        th = np.array([enc(nm, s[nm]) for nm in free])
        prev = np.inf
        for _ in range(4):
            res = minimize(negll, th, method="Nelder-Mead",
                           options=dict(maxiter=maxiter, xatol=1e-2, fatol=0.02, adaptive=True))
            th = res.x
            if prev - res.fun < 0.5:
                prev = min(prev, res.fun)
                break
            prev = res.fun
        if best is None or prev < best[0]:
            best = (prev, th.copy())
    par = dict(base)
    for nm, t in zip(free, best[1]):
        par[nm] = dec(nm, t)
    return par, -best[0], ne[0]


def weibull_fit(times_per_fault, T):
    """Pooled Weibull renewal MLE (shape k, scale s) from inter-event intervals with a right-censored last interval."""
    ints, cens = [], []
    for ts in times_per_fault:
        prev = 0.0
        for t in ts:
            ints.append(t - prev)
            prev = t
        cens.append(T - prev)
    ints, cens = np.array(ints), np.array(cens)
    if len(ints) < 2:
        return 1.0, max(T / max(len(ints), 1), 1.0)

    def nll(th):
        k, s = np.exp(th)
        ll = np.sum(np.log(k / s) + (k - 1) * np.log(ints / s) - (ints / s) ** k) - np.sum((cens / s) ** k)
        return -ll
    r = minimize(nll, np.log([2.0, ints.mean()]), method="Nelder-Mead")
    k, s = np.exp(r.x)
    return float(k), float(s)


def baseline_probs(train_events, T, eval_events, origins, W):
    """Poisson and Weibull-renewal window probabilities for both faults at the origins."""
    rate = np.array([(len(train_events[i]) + 0.5) / T for i in range(2)])
    k, s = weibull_fit(train_events, T)
    p_pois = 1 - np.exp(-rate * W)
    pp = np.tile(p_pois, (len(origins), 1))
    pw = np.zeros((len(origins), 2))
    for i in range(2):
        ev = np.array(eval_events[i])
        for r, t in enumerate(origins):
            prev = ev[ev <= t]
            u = (t - prev[-1]) if len(prev) else t
            pw[r, i] = 1 - np.exp(-((u + W) / s) ** k + (u / s) ** k)
    return pp, pw, (k, s)


def events_from(data):
    return [list(np.where(data["ev"][:, i])[0] * data["h"]) for i in range(2)]


def run_alt(args):
    s, T, npart, out = args
    fn = os.path.join(out, f"alt_T{int(T)}_sy{TAG}_{s:03d}.json")
    if os.path.exists(fn):
        return fn
    t0 = time.time()
    data = rs.generate(TRUE, T=T, h=0.25, seed=1000 + s)
    prep = fp.prepare(data)
    res = dict(rep=s, T=T, n_events=int(data["ev"].sum()))
    ll_true = fp.loglik(TRUE, data, prep, FIXED, npart, 0)
    # H0: no memory (alpha = 1), free gamma, c, b
    base1 = dict(TRUE, alpha=1.0)
    r1, ll1, _ = fit(data, prep, ["gamma", "c", "b"], base1, STARTS, npart)
    near1 = dict(alpha=0.99, gamma=r1["gamma"], c=r1["c"], b=r1["b"])
    full, ll_full, ne = fit(data, prep, NAMES, TRUE, STARTS + [near1], npart)
    res.update(full=full, ll_full=ll_full, ll_true=ll_true, evals=ne, valid=bool(ll_full >= ll_true - 2.0))
    res.update(alpha1=r1, ll_alpha1=ll1)
    # H0: no interaction (gamma = 0), free alpha, b
    base0 = dict(TRUE, gamma=0.0)
    r0, ll0, _ = fit(data, prep, ["alpha", "b"], base0, STARTS, npart)
    res.update(gamma0=r0, ll_gamma0=ll0)
    # profile likelihood over alpha
    grid = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]
    prof = []
    for a in (grid if s < 6 else []):
        basea = dict(full, alpha=a)
        if a == 1.0:
            prof.append((a, ll1))
            continue
        _, lla, _ = fit(data, prep, ["gamma", "c", "b"], basea, [full], npart, maxiter=120)
        prof.append((a, max(lla, -1e8)))
    res["profile"] = prof
    # out-of-sample forecasts on an independent dataset of the same length
    W = 5.0
    ed = rs.generate(TRUE, T=T, h=0.25, seed=5000 + s)
    ep = fp.prepare(ed)
    tf, p_full = fp.forecast(full, ed, ep, FIXED, npart=100, K=4, window_years=W)
    _, p_a1 = fp.forecast(r1, ed, ep, FIXED, npart=100, K=4, window_years=W)
    tev = events_from(data)
    eev = events_from(ed)
    p_pois, p_weib, wk = baseline_probs(tev, T, eev, tf, W)
    y = np.zeros((len(tf), 2))
    for i in range(2):
        e = np.array(eev[i])
        for r, t in enumerate(tf):
            y[r, i] = float(np.any((e > t) & (e <= t + W)))
    np.savez(fn.replace(".json", ".npz"), t=tf, y=y, p_full=p_full, p_alpha1=p_a1, p_pois=p_pois, p_weib=p_weib)
    res["weibull"] = wk
    res["seconds"] = time.time() - t0
    with open(fn, "w") as f:
        json.dump(res, f)
    return fn


def run_null(args):
    s, T, npart, out = args
    fn = os.path.join(out, f"null_T{int(T)}_sy{TAG}_{s:03d}.json")
    if os.path.exists(fn):
        return fn
    t0 = time.time()
    par0 = dict(TRUE, alpha=1.0)
    data = rs.generate(dict(par0), T=T, h=0.25, seed=3000 + s)
    prep = fp.prepare(data)
    r1, ll1, _ = fit(data, prep, ["gamma", "c", "b"], par0, STARTS, npart)
    near1 = dict(alpha=0.99, gamma=r1["gamma"], c=r1["c"], b=r1["b"])
    full, ll_full, _ = fit(data, prep, NAMES, par0, STARTS + [near1], npart)
    res = dict(rep=s, T=T, n_events=int(data["ev"].sum()), full=full, ll_full=ll_full, ll_alpha1=ll1,
               lr=max(2 * (ll_full - ll1), 0.0), seconds=time.time() - t0)
    with open(fn, "w") as f:
        json.dump(res, f)
    return fn


def dispatch(a):
    kind = a[0]
    return run_alt(a[1:]) if kind == "alt" else run_null(a[1:])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--T", type=float, default=400.0)
    ap.add_argument("--R", type=int, default=30)
    ap.add_argument("--B0", type=int, default=60)
    ap.add_argument("--npart", type=int, default=100)
    ap.add_argument("--procs", type=int, default=2)
    ap.add_argument("--out", default=os.path.join(HERE, "out"))
    ap.add_argument("--only", default="both")
    ap.add_argument("--sigy", type=float, default=0.3)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rs.FIXED["sigy"] = a.sigy
    TAG = str(a.sigy)
    tasks = []
    if a.only in ("both", "alt"):
        tasks += [("alt", s, a.T, a.npart, a.out) for s in range(a.R)]
    if a.only in ("both", "null"):
        tasks += [("null", s, a.T, a.npart, a.out) for s in range(a.B0)]
    t0 = time.time()
    with Pool(a.procs) as pool:
        for i, fn in enumerate(pool.imap_unordered(dispatch, tasks), 1):
            print(f"[{i}/{len(tasks)}] {os.path.basename(fn)}  elapsed {time.time()-t0:.0f}s", flush=True)
    print("DONE", flush=True)


In [ ]:
%%writefile misspec.py
"""Robustness to wrong reference recurrence times: refit replicate datasets with Tbar scaled by 0.8 and 1.25."""
import json, os, sys, time
import numpy as np
from multiprocessing import Pool

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import stats_study as ss
fp, rs = ss.fp, ss.rs


def one(args):
    s, scale = args
    fx = fp.FIXED
    true_T = np.array([40.0, 45.0])
    data = rs.generate(ss.TRUE, T=400.0, h=0.25, seed=1000 + s)       # data always generated with the true Tbar
    prep = fp.prepare(data)
    fx["Tbar"] = true_T * scale                                          # inference uses the wrong reference times
    par, ll, _ = ss.fit(data, prep, ss.NAMES, ss.TRUE, ss.STARTS, 100)
    fx["Tbar"] = true_T
    ll_ok = fp.loglik(ss.TRUE, data, prep, fx, 100, 0)
    return dict(rep=s, scale=scale, par=par, ll=ll, ll_truth_correct_T=ll_ok)


if __name__ == "__main__":
    R = int(sys.argv[1]) if len(sys.argv) > 1 else 12
    tasks = [(s, sc) for s in range(R) for sc in (0.8, 1.25)]
    t0 = time.time()
    with Pool(2) as pool:
        res = []
        for i, r in enumerate(pool.imap_unordered(one, tasks), 1):
            res.append(r)
            print(i, len(tasks), f"{time.time()-t0:.0f}s", flush=True)
    json.dump(res, open(os.path.join(HERE, "out", "misspec.json"), "w"))
    print("DONE")


In [ ]:
%%writefile fig_basic.py
import os, sys, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from math import gamma as Gamma

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import fastpf as fp
rs = fp.rs
warnings.filterwarnings("ignore")
FIG = os.path.join(HERE, "figs")
os.makedirs(FIG, exist_ok=True)
C1, C2, CG, CK = "#0072B2", "#D55E00", "#009E73", "#333333"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "pdf.fonttype": 42, "figure.dpi": 150,
                     "legend.frameon": False, "axes.titlesize": 9.5})
TRUE, FIXED = fp.TRUE, fp.FIXED

# ---------------------------------------------------------------- Fig: simulated dataset (seed 11)
data = rs.generate(TRUE, T=400.0, h=0.25, seed=11)
t = np.arange(data["n"] + 1) * data["h"]
x = data["x"]
ts = data["obs_steps"] * data["h"]
fig, axs = plt.subplots(2, 1, figsize=(7.2, 4.6), sharex=True)
for i, (ax, col) in enumerate(zip(axs, (C1, C2))):
    ax.plot(t, x[:, i], color=col, lw=1.2, label="latent stress $x_i(t)$")
    ax.plot(ts, data["y"][:, i], ".", color=CK, ms=2, alpha=0.5, label="yearly observation $y_i$")
    ax.axhline(FIXED["theta"], color=CG, ls="--", lw=1, label=r"threshold $\theta$")
    ev = np.where(data["ev"][:, i])[0] * data["h"]
    ax.plot(ev, np.full(len(ev), FIXED["theta"] + 1.2), "v", color=col, ms=7, label="rupture")
    for e in ev:  # delayed pulse arrival on the other fault
        pass
    ax.set_ylabel(f"stress, fault {i+1}")
    ax.set_ylim(0, 12.5)
axs[0].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.28), fontsize=8)
axs[1].set_xlabel("time (years)")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "sim_dataset.pdf"))
plt.close(fig)

# ---------------------------------------------------------------- Fig: kernel and SOE approximation
h, T = 0.25, 400.0
lags = np.arange(1, int(T / h) + 1) * h
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9))
for a, col in zip((0.5, 0.7, 0.9), (C1, C2, CG)):
    s, w = rs.soe(a, h, T)
    ex = lags ** (a - 1) / Gamma(a)
    ap = (w[None, :] * np.exp(-s[None, :] * lags[:, None])).sum(1)
    axs[0].loglog(lags, ex, color=col, lw=1.6, label=fr"exact, $\alpha={a}$")
    axs[0].loglog(lags, ap, color=CK, lw=0.8, ls=":")
    axs[1].semilogx(lags, 100 * (ap - ex) / ex, color=col, lw=1.2, label=fr"$\alpha={a}$")
axs[0].set_xlabel("lag (years)"); axs[0].set_ylabel(r"kernel $t^{\alpha-1}/\Gamma(\alpha)$")
axs[0].set_title("Kernel (solid) and sum of exponentials (dotted)")
axs[0].legend(fontsize=7.5)
axs[1].set_xlabel("lag (years)"); axs[1].set_ylabel("relative error (%)")
axs[1].set_title("Pointwise error of the approximation")
axs[1].axhline(0, color="k", lw=0.6)
axs[1].legend(fontsize=7.5)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "soe_kernel.pdf"))
plt.close(fig)

# ---------------------------------------------------------------- Fig: loading calibration, isolated fault
fig, ax = plt.subplots(figsize=(4.8, 3.0))
fx = dict(FIXED, sigx=0.0)
tt = np.linspace(0, 60, 601)
for a, col in zip((1.0, 0.7, 0.5), (CK, C1, C2)):
    v = rs.loading_rates(a, fx)[0]
    E = np.array([rs.mittag_leffler(a, -fx["beta"] * q ** a) for q in tt])
    xx = v / fx["beta"] * (1 - E) + fx["rho"] * fx["theta"] * E
    ax.plot(tt, xx, color=col, lw=1.6, label=fr"$\alpha={a}$, $v={v:.2f}$")
ax.axhline(fx["theta"], color=CG, ls="--", lw=1)
ax.axvline(40, color=CG, ls="--", lw=1)
ax.text(41, 10.4, r"$\bar T=40$", color=CG)
ax.set_xlabel("time since rupture (years)"); ax.set_ylabel("stress of an isolated fault")
ax.set_ylim(0, 12); ax.legend(fontsize=8, loc="lower right", bbox_to_anchor=(1.0, 0.02))
fig.tight_layout()
fig.savefig(os.path.join(FIG, "loading_calibration.pdf"))
plt.close(fig)

# ---------------------------------------------------------------- Fig: likelihood slices (dataset seed 11)
prep = fp.prepare(data)
grids = dict(alpha=np.linspace(0.64, 0.76, 49), gamma=np.linspace(0.05, 0.12, 36),
             c=30.0/np.arange(1.0, 6.01, 0.25), b=np.exp(np.linspace(np.log(1.0), np.log(10), 40)))
labels = dict(alpha=r"$\alpha$", gamma=r"$\gamma$", c=r"delay $\tau=d/c$ (yr)", b=r"$b$")
fig, axs = plt.subplots(1, 4, figsize=(7.4, 2.5))
P = 200
for ax, nm in zip(axs, ("alpha", "gamma", "c", "b")):
    curves = []
    for sd in range(4):
        base = fp.loglik(TRUE, data, prep, FIXED, P, sd)
        ll = np.array([fp.loglik(dict(TRUE, **{nm: g}), data, prep, FIXED, P, sd) for g in grids[nm]])
        curves.append(ll - base)
    curves = np.array(curves)
    m = curves.mean(0)
    xg = 30.0 / grids[nm] if nm == "c" else grids[nm]
    xt = 3.0 if nm == "c" else TRUE[nm]
    ax.plot(xg, m, color=C1, lw=1.4, marker="o" if nm == "c" else None, ms=3)
    ax.fill_between(xg, curves.min(0), curves.max(0), color=C1, alpha=0.2, lw=0)
    ax.axvline(xt, color=C2, ls="--", lw=1)
    ax.set_xlabel(labels[nm]); ax.set_ylim(-45, 8)
    if nm == "b":
        ax.set_xscale("log")
axs[0].set_ylabel(r"$\ell(\theta)-\ell(\theta_0)$")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "slices.pdf"))
plt.close(fig)
print("basic figures done")


In [ ]:
%%writefile analyze.py
import glob, json, os, sys, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import rankdata

warnings.filterwarnings("ignore")
HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(HERE, "out")
FIG = os.path.join(HERE, "figs")
TAB = os.path.join(HERE, "tabs")
os.makedirs(TAB, exist_ok=True)
C1, C2, CG, CK, CP = "#0072B2", "#D55E00", "#009E73", "#333333", "#CC79A7"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "pdf.fonttype": 42, "figure.dpi": 150, "legend.frameon": False,
                     "axes.titlesize": 9.5})
TRUE = dict(alpha=0.7, gamma=0.08, c=10.0, b=3.0)
SC = [("0.3", r"$\sigma_y=0.3$"), ("1.5", r"$\sigma_y=1.5$")]
W = 5.0
S = {}


def load(kind, sy):
    rows = []
    for f in sorted(glob.glob(os.path.join(OUT, f"{kind}_T400_sy{sy}_*.json"))):
        rows.append(json.load(open(f)))
    return rows


def auc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(p)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def brier(y, p):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def logscore(y, p, eps=1e-3):
    p = np.clip(np.asarray(p), eps, 1 - eps); y = np.asarray(y)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


have = {}
for sy, _ in SC:
    have[sy] = (load("alt", sy), load("null", sy))
    print(sy, "alt", len(have[sy][0]), "null", len(have[sy][1]))

# ------------------------------------------------------------------ recovery table + figure
fig, axs = plt.subplots(1, 4, figsize=(7.4, 2.8))
names = ["alpha", "gamma", "c", "b"]
tex = [r"\begin{tabular}{llrrrrrr}", r"\toprule",
       r"$\sigma_y$ & Parameter & Truth & Mean & Bias & SD & RMSE & Rel.\ RMSE (\%)\\", r"\midrule"]
for k, (sy, lab) in enumerate(SC):
    alt, _ = have[sy]
    ok = [r for r in alt if r["valid"]]
    S[f"n_alt_{sy}"] = len(alt); S[f"n_valid_{sy}"] = len(ok)
    for j, nm in enumerate(names):
        est = np.array([r["full"][nm] for r in ok])
        if len(est) == 0:
            continue
        tr = TRUE[nm]
        bias = est.mean() - tr; sd = est.std(ddof=1) if len(est) > 1 else np.nan
        rmse = np.sqrt(np.mean((est - tr) ** 2))
        S[f"{nm}_{sy}"] = dict(mean=est.mean(), bias=bias, sd=sd, rmse=rmse, rel=100 * rmse / tr,
                               med=float(np.median(est)))
        sym = {"alpha": r"$\alpha$", "gamma": r"$\gamma$", "c": r"$c$", "b": r"$b$"}[nm]
        first = lab if j == 0 else ""
        tex.append(f"{first} & {sym} & {tr:g} & {est.mean():.3f} & {bias:+.3f} & {sd:.3f} & {rmse:.3f} & {100*rmse/tr:.1f}\\\\")
        ax = axs[j]
        pos = k + 1
        ax.boxplot(est / tr, positions=[pos], widths=0.5, showfliers=False,
                   medianprops=dict(color=CK), boxprops=dict(color=CK), whiskerprops=dict(color=CK),
                   capprops=dict(color=CK))
        ax.plot(pos + 0.0 + (np.random.default_rng(1).random(len(est)) - 0.5) * 0.3, est / tr, ".",
                color=[C1, C2][k], ms=4, alpha=0.7)
    if sy != SC[-1][0]:
        tex.append(r"\midrule")
tex += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB, "tab_recovery.tex"), "w").write("\n".join(tex))
for j, nm in enumerate(names):
    ax = axs[j]
    ax.axhline(1.0, color=CG, ls="--", lw=1)
    ax.set_xticks([1, 2]); ax.set_xticklabels([r"0.3", r"1.5"])
    ax.set_xlabel(r"$\sigma_y$")
    ax.set_title({"alpha": r"$\hat\alpha/\alpha$", "gamma": r"$\hat\gamma/\gamma$", "c": r"$\hat c/c$", "b": r"$\hat b/b$"}[nm])
fig.tight_layout()
fig.savefig(os.path.join(FIG, "recovery_box.pdf"))
plt.close(fig)

# ------------------------------------------------------------------ profile deviance over alpha
alt01 = have["0.3"][0]
fig, ax = plt.subplots(figsize=(4.8, 3.0))
for r in alt01:
    if r.get("profile"):
        a = np.array([p[0] for p in r["profile"]]); l = np.array([p[1] for p in r["profile"]])
        dev = 2 * (r["ll_full"] - l)
        ax.plot(a, np.clip(dev, 0, 400), "-o", ms=2.5, lw=1, alpha=0.8)
ax.axvline(0.7, color=CG, ls="--", lw=1)
ax.axhline(3.84, color=CK, ls=":", lw=1)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"profile deviance $2(\hat\ell-\ell_p(\alpha))$ (clipped at 400)")
ax.set_ylim(0, 400)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "profile_alpha.pdf"))
plt.close(fig)

# ------------------------------------------------------------------ LR tests for memory and interaction
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9))
tex = [r"\begin{tabular}{lrrrrrr}", r"\toprule",
       r"$\sigma_y$ & $B_0$ & $\hat\alpha<0.98$ under $H_0$ (\%) & Type I, $\chi$-ref.\ (\%) & Crit.\ (95\,\%, boot.) & Power, memory (\%) & Power, coupling (\%)\\",
       r"\midrule"]
for k, (sy, lab) in enumerate(SC):
    alt, null = have[sy]
    lr0 = np.array([r["lr"] for r in null])
    ok = [r for r in alt if r["valid"]]
    lra = np.array([max(2 * (r["ll_full"] - r["ll_alpha1"]), 0) for r in ok])
    lrg = np.array([max(2 * (r["ll_full"] - r["ll_gamma0"]), 0) for r in ok])
    if len(lr0) == 0 or len(lra) == 0:
        continue
    crit_chi = 2.706
    crit_boot = float(np.quantile(lr0, 0.95))
    S[f"lr_{sy}"] = dict(B0=len(lr0), type1=float(np.mean(lr0 > crit_chi)), crit=crit_boot,
                         power=float(np.mean(lra > max(crit_boot, 1e-9))), power_g=float(np.mean(lrg > crit_chi)),
                         alpha_null_mean=float(np.mean([r["full"]["alpha"] for r in null])),
                         frac_a98=float(np.mean([r["full"]["alpha"] < 0.98 for r in null])),
                         lra_min=float(lra.min()), lra_med=float(np.median(lra)))
    tex.append(f"{lab} & {len(lr0)} & {100*S[f'lr_{sy}']['frac_a98']:.0f} & {100*S[f'lr_{sy}']['type1']:.1f} & {crit_boot:.2f} & "
               f"{100*S[f'lr_{sy}']['power']:.0f} & {100*S[f'lr_{sy}']['power_g']:.0f}\\\\")
    x = np.sort(lr0); y = np.arange(1, len(x) + 1) / len(x)
    axs[0].step(np.r_[0, x], np.r_[0, y], where="post", color=[C1, C2][k], label=f"null, {lab}")
    axs[1].plot(np.sort(lra), np.arange(1, len(lra) + 1) / len(lra), "-", color=[C1, C2][k], label=f"true $\\alpha=0.7$, {lab}")
tex += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB, "tab_tests.tex"), "w").write("\n".join(tex))
xs = np.linspace(0, 8, 200)
from scipy.stats import chi2
axs[0].plot(xs, 0.5 + 0.5 * chi2.cdf(xs, 1), color=CK, ls="--", lw=1, label=r"$0.5\chi^2_0+0.5\chi^2_1$")
axs[0].set_xlim(0, 8); axs[0].set_xlabel("LR statistic"); axs[0].set_ylabel("empirical CDF")
axs[0].set_title(r"Null law of the LR statistic, $H_0:\alpha=1$"); axs[0].legend(fontsize=7)
axs[1].set_xscale("symlog", linthresh=10); axs[1].set_xlabel("LR statistic (log scale)")
axs[1].axvline(2.706, color=CK, ls=":", lw=1)
axs[1].set_title("Under the alternative"); axs[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "lr_tests.pdf"))
plt.close(fig)

# ------------------------------------------------------------------ forecast evaluation
methods = [("p_full", "fractional (fitted)", C1), ("p_alpha1", "no memory ($\\alpha=1$)", C2),
           ("p_weib", "Weibull renewal", CG), ("p_pois", "Poisson", CK)]
fig, axs = plt.subplots(1, 3, figsize=(7.6, 2.9))
tex = [r"\begin{tabular}{llrrrr}", r"\toprule",
       r"$\sigma_y$ & Method & Brier & Log score & AUC & Brier skill vs.\ Poisson (\%)\\", r"\midrule"]
bss_data = {}
for k, (sy, lab) in enumerate(SC):
    fcs = [np.load(f) for f in sorted(glob.glob(os.path.join(OUT, f"alt_T400_sy{sy}_*.npz")))]
    meta = {json.load(open(f))["rep"]: json.load(open(f)) for f in glob.glob(os.path.join(OUT, f"alt_T400_sy{sy}_*.json"))}
    if not fcs:
        continue
    pooled = {m: [] for m, _, _ in methods}; Y = []
    per_rep = {m: [] for m, _, _ in methods}
    for d in fcs:
        keep = d["t"] <= 400.0 - W
        y = d["y"][keep].ravel()
        Y.append(y)
        for m, _, _ in methods:
            p = d[m][keep].ravel()
            pooled[m].append(p)
            per_rep[m].append(brier(y, p))
    Y = np.concatenate(Y)
    S[f"fc_{sy}"] = dict(n=len(fcs), npairs=int(len(Y)), pos=int(Y.sum()))
    bp = np.array(per_rep["p_pois"])
    for m, nm, col in methods:
        p = np.concatenate(pooled[m])
        b = brier(Y, p); ls = logscore(Y, p); a = auc(Y, p)
        bss = 1 - np.array(per_rep[m]) / bp
        bss_data[(sy, m)] = (bss.mean() * 100, bss.std(ddof=1) / np.sqrt(len(bss)) * 100 if len(bss) > 1 else 0)
        S[f"fc_{sy}"][m] = dict(brier=b, logscore=ls, auc=a, bss=bss.mean() * 100)
        tex.append(f"{lab if m == 'p_full' else ''} & {nm} & {b:.4f} & {ls:.4f} & {a:.3f} & {bss.mean()*100:.1f}\\\\")
        if sy == "0.3":
            # reliability (deciles of forecast)
            qs = np.unique(np.quantile(p, np.linspace(0, 1, 9)))
            idx = np.clip(np.digitize(p, qs[1:-1]), 0, len(qs) - 2)
            xm = [p[idx == i].mean() for i in range(len(qs) - 1) if (idx == i).any()]
            ym = [Y[idx == i].mean() for i in range(len(qs) - 1) if (idx == i).any()]
            axs[0].plot(xm, ym, "-o", ms=3, color=col, label=nm, lw=1.2)
            # ROC
            thr = np.unique(np.r_[0, p, 1])
            P1 = Y.sum(); P0 = len(Y) - P1
            order = np.argsort(-p)
            tp = np.cumsum(Y[order]) / P1; fpr = np.cumsum(1 - Y[order]) / P0
            axs[1].plot(np.r_[0, fpr], np.r_[0, tp], color=col, lw=1.2, label=nm)
    if sy != SC[-1][0]:
        tex.append(r"\midrule")
tex += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB, "tab_forecast.tex"), "w").write("\n".join(tex))
axs[0].plot([0, 0.6], [0, 0.6], color="k", lw=0.6, ls=":")
axs[0].set_xlabel("forecast probability"); axs[0].set_ylabel("observed frequency")
axs[0].set_title(r"Reliability ($\sigma_y=0.3$)"); axs[0].legend(fontsize=6.5)
axs[1].plot([0, 1], [0, 1], color="k", lw=0.6, ls=":")
axs[1].set_xlabel("false-positive rate"); axs[1].set_ylabel("true-positive rate"); axs[1].set_title("ROC")
width = 0.2
for mi, (m, nm, col) in enumerate(methods[:3]):
    vals = [bss_data.get((sy, m), (np.nan, 0)) for sy, _ in SC]
    axs[2].bar(np.arange(2) + (mi - 1) * width, [v[0] for v in vals], width, yerr=[v[1] for v in vals], color=col, label=nm,
               capsize=2)
axs[2].set_xticks([0, 1]); axs[2].set_xticklabels([r"$\sigma_y=0.3$", r"$\sigma_y=1.5$"])
axs[2].axhline(0, color="k", lw=0.6)
axs[2].set_ylabel("Brier skill vs. Poisson (%)"); axs[2].set_title("Skill (mean $\\pm$ SE over replicates)")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "forecast_eval.pdf"))
plt.close(fig)

# ------------------------------------------------------------------ misspecified reference time (needs misspec.py)
mf = os.path.join(OUT, "misspec.json")
if os.path.exists(mf):
    ms = json.load(open(mf))
    base = {r["rep"]: r for r in have["0.3"][0]}
    tex = [r"\begin{tabular}{lrrrrr}", r"\toprule",
           r"$\bar T$ scale & $\bar\alpha$ & $\bar\gamma$ & $\bar c$ & $\bar b$ & Mean log-lik.\ loss\\", r"\midrule"]
    for sc in (0.8, 1.25):
        rr = [r for r in ms if r["scale"] == sc and r["rep"] in base]
        if not rr:
            continue
        loss = np.mean([base[r["rep"]]["ll_full"] - r["ll"] for r in rr])
        m = {n: np.mean([r["par"][n] for r in rr]) for n in names}
        tex.append(f"{sc:g} & {m['alpha']:.3f} & {m['gamma']:.3f} & {m['c']:.2f} & {m['b']:.2f} & {loss:.1f}\\\\")
    tex += [r"\bottomrule", r"\end{tabular}"]
    open(os.path.join(TAB, "tab_misspec.tex"), "w").write("\n".join(tex))

json.dump(S, open(os.path.join(HERE, "summary.json"), "w"), indent=1, default=float)
print(json.dumps(S, indent=1, default=float))


In [ ]:
%%writefile fault_map.py
"""Real map of two neighbouring faults (GEM Global Active Faults Database) with cartopy.
Run in Colab (needs internet + cartopy). Writes figs/fault_map.pdf/.png and out_fault_pair.json.
Change REGION / NAME1 / NAME2 to choose another pair of faults (the printed table lists candidates)."""
import json, os, sys, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

HERE = os.path.dirname(os.path.abspath(__file__))
FIG = os.path.join(HERE, "figs"); os.makedirs(FIG, exist_ok=True)
DATA = os.path.join(HERE, "data"); os.makedirs(DATA, exist_ok=True)
URLS = ["https://raw.githubusercontent.com/cossatot/gem-global-active-faults/master/geojson/gem_active_faults.geojson",
        "https://raw.githubusercontent.com/GEMScienceTools/gem-global-active-faults/master/geojson/gem_active_faults.geojson"]
# ---- user choices ------------------------------------------------------------------------------------
REGION = (43.0, 51.0, 35.5, 40.5)      # lon_min, lon_max, lat_min, lat_max  (north-west Iran, around Tabriz)
NAME1 = os.environ.get("FAULT1", "Tabriz")  # substring of the name of fault 1 (None = longest fault in the box)
NAME2 = os.environ.get("FAULT2", "")        # substring of fault 2 ("" = nearest other fault longer than MINLEN)
MINLEN = 40.0                          # km
# ------------------------------------------------------------------------------------------------------


def haversine(lon1, lat1, lon2, lat2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def load():
    fn = os.path.join(DATA, "gem_active_faults.geojson")
    if not os.path.exists(fn):
        for u in URLS:
            try:
                print("downloading", u)
                urllib.request.urlretrieve(u, fn)
                break
            except Exception as e:
                print("failed:", e)
    return json.load(open(fn))


def lines_of(geom):
    if geom["type"] == "LineString":
        return [np.array(geom["coordinates"])[:, :2]]
    if geom["type"] == "MultiLineString":
        return [np.array(c)[:, :2] for c in geom["coordinates"]]
    return []


gj = load()
lo0, lo1, la0, la1 = REGION
F = []
for ft in gj["features"]:
    ls = lines_of(ft["geometry"])
    ls = [l for l in ls if len(l) > 1 and (l[:, 0] > lo0).any() and (l[:, 0] < lo1).any() and (l[:, 1] > la0).any() and (l[:, 1] < la1).any()]
    if not ls:
        continue
    pr = ft.get("properties", {})
    name = str(pr.get("name") or pr.get("catalog_name") or pr.get("fault_name") or "unnamed")
    length = sum(haversine(l[:-1, 0], l[:-1, 1], l[1:, 0], l[1:, 1]).sum() for l in ls)
    F.append(dict(name=name, lines=ls, length=length, slip=pr.get("slip_type", "")))
print(f"{len(F)} fault traces in the box; longest ones:")
for f in sorted(F, key=lambda f: -f["length"])[:25]:
    print(f"  {f['name'][:50]:50s} {f['length']:7.1f} km  {f['slip']}")


def pts(f):
    p = np.vstack(f["lines"])
    return p[:: max(1, len(p) // 1500)]


def mind(f, g):
    a, b = pts(f), pts(g)
    D = haversine(a[:, None, 0], a[:, None, 1], b[None, :, 0], b[None, :, 1])
    i, j = np.unravel_index(D.argmin(), D.shape)
    return D[i, j], a[i], b[j]


c1 = [f for f in F if NAME1 and NAME1.lower() in f["name"].lower()] or sorted(F, key=lambda f: -f["length"])[:1]
f1 = max(c1, key=lambda f: f["length"])
if NAME2:
    f2 = max([f for f in F if NAME2.lower() in f["name"].lower()], key=lambda f: f["length"])
else:
    cand = [f for f in F if f is not f1 and f["name"] != f1["name"] and f["length"] >= MINLEN]
    f2 = min(cand, key=lambda f: mind(f1, f)[0])
d, p1, p2 = mind(f1, f2)
print(f"\nFault 1: {f1['name']} ({f1['length']:.0f} km)\nFault 2: {f2['name']} ({f2['length']:.0f} km)\nclosest approach d = {d:.1f} km")
json.dump(dict(fault1=f1["name"], length1_km=f1["length"], fault2=f2["name"], length2_km=f2["length"], d_km=float(d),
               source="GEM Global Active Faults Database (Styron & Pagani, 2020)"),
          open(os.path.join(HERE, "out_fault_pair.json"), "w"), indent=1)

fig = plt.figure(figsize=(7.4, 5.6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lo0, lo1, la0, la1], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#f2efe6")
ax.add_feature(cfeature.OCEAN, facecolor="#dbe9f4")
ax.add_feature(cfeature.LAKES, facecolor="#dbe9f4", edgecolor="#7aa6c9", lw=0.4)
ax.add_feature(cfeature.BORDERS, lw=0.6, ls="--", edgecolor="#777777")
ax.add_feature(cfeature.COASTLINE, lw=0.6)
gl = ax.gridlines(draw_labels=True, lw=0.3, color="gray", alpha=0.6)
gl.top_labels = gl.right_labels = False
for f in F:
    for l in f["lines"]:
        ax.plot(l[:, 0], l[:, 1], color="#999999", lw=0.6, transform=ccrs.PlateCarree(), zorder=2)
for f, col, lab in ((f1, "#0072B2", "Fault 1"), (f2, "#D55E00", "Fault 2")):
    for k, l in enumerate(f["lines"]):
        ax.plot(l[:, 0], l[:, 1], color=col, lw=2.6, transform=ccrs.PlateCarree(), zorder=4,
                label=f"{lab}: {f['name']} ({f['length']:.0f} km)" if k == 0 else None)
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="k", ls="--", lw=1.2, transform=ccrs.PlateCarree(), zorder=5)
ax.text((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2, f"  $d\\approx{d:.0f}$ km", transform=ccrs.PlateCarree(), zorder=6,
        fontsize=9, bbox=dict(fc="white", ec="none", alpha=0.8, pad=1))
for nm, lon, lat in (("Tabriz", 46.29, 38.08), ("Tehran", 51.39, 35.69), ("Urmia", 45.07, 37.55)):
    if lo0 < lon < lo1 and la0 < lat < la1:
        ax.plot(lon, lat, "o", color="k", ms=3.5, transform=ccrs.PlateCarree(), zorder=6)
        ax.text(lon + 0.07, lat + 0.05, nm, fontsize=7.5, transform=ccrs.PlateCarree(), zorder=6)
ax.legend(loc="lower left", fontsize=7.5, framealpha=0.9)
ax.set_title("Two neighbouring active faults (real traces)  --  GEM Global Active Faults Database", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fault_map.pdf"))
fig.savefig(os.path.join(FIG, "fault_map.png"), dpi=170)
print("saved figs/fault_map.pdf / .png")


In [ ]:
%%writefile export_excel.py
"""Collect all study results in one Excel workbook (results.xlsx) + the figures stay in figs/."""
import glob, json, os
import numpy as np
import pandas as pd
HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(HERE, "out")
rows_alt, rows_null = [], []
for f in sorted(glob.glob(os.path.join(OUT, "alt_*.json"))):
    r = json.load(open(f)); sy = os.path.basename(f).split("_sy")[1].split("_")[0]
    rows_alt.append(dict(sigma_y=float(sy), rep=r["rep"], n_events=r["n_events"], valid=r["valid"], ll_full=r["ll_full"],
                         ll_true=r["ll_true"], ll_alpha1=r["ll_alpha1"], ll_gamma0=r["ll_gamma0"],
                         LR_memory=max(2 * (r["ll_full"] - r["ll_alpha1"]), 0), LR_coupling=max(2 * (r["ll_full"] - r["ll_gamma0"]), 0),
                         alpha_hat=r["full"]["alpha"], gamma_hat=r["full"]["gamma"], c_hat=r["full"]["c"], b_hat=r["full"]["b"],
                         weibull_shape=r["weibull"][0], weibull_scale=r["weibull"][1], seconds=r["seconds"]))
for f in sorted(glob.glob(os.path.join(OUT, "null_*.json"))):
    r = json.load(open(f)); sy = os.path.basename(f).split("_sy")[1].split("_")[0]
    rows_null.append(dict(sigma_y=float(sy), rep=r["rep"], n_events=r["n_events"], LR=r["lr"], alpha_hat=r["full"]["alpha"],
                          ll_full=r["ll_full"], ll_alpha1=r["ll_alpha1"]))
alt, null = pd.DataFrame(rows_alt), pd.DataFrame(rows_null)
TRUE = dict(alpha_hat=0.7, gamma_hat=0.08, c_hat=10.0, b_hat=3.0)
rec = []
for sy, g in alt[alt.valid].groupby("sigma_y"):
    for k, t in TRUE.items():
        e = g[k].values
        rec.append(dict(sigma_y=sy, parameter=k.replace("_hat", ""), truth=t, n=len(e), mean=e.mean(), bias=e.mean() - t,
                        sd=e.std(ddof=1) if len(e) > 1 else np.nan, rmse=np.sqrt(np.mean((e - t) ** 2)), rel_rmse_pct=100 * np.sqrt(np.mean((e - t) ** 2)) / t))
rec = pd.DataFrame(rec)
tests = []
for sy in sorted(set(null.sigma_y) & set(alt.sigma_y)):
    n0, a = null[null.sigma_y == sy], alt[(alt.sigma_y == sy) & alt.valid]
    crit = np.quantile(n0.LR, 0.95)
    tests.append(dict(sigma_y=sy, B0=len(n0), crit95_bootstrap=crit, size_chi2_2706=(n0.LR > 2.706).mean(),
                      power_memory=(a.LR_memory > max(crit, 1e-9)).mean(), power_coupling_chi2=(a.LR_coupling > 2.706).mean()))
tests = pd.DataFrame(tests)
with pd.ExcelWriter(os.path.join(HERE, "results.xlsx")) as w:
    rec.to_excel(w, sheet_name="Recovery_summary", index=False)
    tests.to_excel(w, sheet_name="LR_tests", index=False)
    alt.to_excel(w, sheet_name="Replicates_alt", index=False)
    null.to_excel(w, sheet_name="Replicates_null", index=False)
    ms = os.path.join(OUT, "misspec.json")
    if os.path.exists(ms):
        pd.DataFrame([dict(rep=r["rep"], Tbar_scale=r["scale"], ll=r["ll"], ll_truth_correct_T=r["ll_truth_correct_T"], **r["par"]) for r in json.load(open(ms))]).to_excel(w, sheet_name="Misspecified_Tbar", index=False)
    fp = os.path.join(HERE, "out_fault_pair.json")
    if os.path.exists(fp):
        pd.DataFrame([json.load(open(fp))]).to_excel(w, sheet_name="Real_fault_pair", index=False)
print(rec.round(3).to_string()); print(tests.round(3).to_string())
print("wrote results.xlsx")


In [ ]:
# 3. size of the study
MODE = 'QUICK'   # 'QUICK' or 'FULL'
CFG = dict(QUICK=dict(R1=4,B1=8,R2=4,B2=8,M=3), FULL=dict(R1=24,B1=40,R2=16,B2=24,M=12))[MODE]
PROCS = multiprocessing.cpu_count()
CFG, PROCS

In [ ]:
# 4. REAL FAULTS: map of two neighbouring faults (cartopy + GEM database)
#    Choose other faults by name (substring), e.g. FAULT1='North Anatolian', FAULT2='East Anatolian' and change REGION in fault_map.py
import os
os.environ['FAULT1'] = 'Tabriz'   # '' = longest fault in the region
os.environ['FAULT2'] = ''         # '' = nearest other fault longer than 40 km
!python fault_map.py

In [ ]:
from IPython.display import Image, display
display(Image('figs/fault_map.png'))

In [ ]:
# 5. basic figures (simulated data, kernel, loading calibration, likelihood slices)
!python fig_basic.py

In [ ]:
# 6. scenario S1 (sigma_y = 0.3)
!python stats_study.py --sigy 0.3 --R {CFG['R1']} --B0 {CFG['B1']} --npart 100 --procs {PROCS} --out out

In [ ]:
# 7. scenario S2 (sigma_y = 1.5)
!python stats_study.py --sigy 1.5 --R {CFG['R2']} --B0 {CFG['B2']} --npart 100 --procs {PROCS} --out out

In [ ]:
# 8. robustness to a wrong reference recurrence time
!python misspec.py {CFG['M']}

In [ ]:
# 9. tables, result figures, Excel workbook (results.xlsx)
!python analyze.py
!python export_excel.py
!cp summary.json results.xlsx {DRIVE}/ 2>/dev/null; true

In [ ]:
# 10. PNG previews of every figure (saved in Drive too)
!for f in figs/*.pdf; do pdftoppm -r 90 -png -singlefile $f ${f%.pdf}; done
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figs/*.png')): display(Image(f))
print('Everything is in', DRIVE)